# 🛒 Retail-Héros — Entraînement GPU sur Google Colab

> **Entraîne ton modèle YOLO pour la détection de produits en rayon — GRATUITEMENT sur GPU Tesla T4**

Ce notebook te permet de :
1. ✅ Vérifier l'accès GPU
2. 📥 Télécharger le dataset SKU110K (11,743 images)
3. 🏋️ Entraîner YOLOv8 sur GPU
4. 📊 Visualiser les résultats
5. 💾 Télécharger le modèle entraîné pour Retail-Héros

**Durée estimée** : 1-2 heures pour 100 epochs

---

## 🚀 Étape 0 : Vérifier le GPU

Exécute cette cellule pour vérifier que Colab t'a attribué un GPU.

> 💡 **Astuce** : Si tu n'as pas de GPU, va dans `Exécution > Modifier le type d'exécution` et sélectionne `T4 GPU`.

In [ ]:
!nvidia-smi

import torch
print(f"\n🔥 PyTorch version: {torch.__version__}")
print(f"🎮 GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 Mémoire GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("❌ PAS DE GPU DÉTECTÉ ! Va dans Exécution > Modifier le type d'exécution > GPU T4")

## 📦 Étape 1 : Installer les dépendances

Installation d'Ultralytics YOLOv8 et des outils nécessaires.

In [ ]:
# Installation d'Ultralytics
!pip install -q ultralytics

# Vérification
import ultralytics
print(f"✅ Ultralytics installé: {ultralytics.__version__}")

## 📥 Étape 2 : Télécharger le dataset SKU110K

YOLO télécharge automatiquement SKU110K (~13.6 GB) au premier entraînement.

> ⏱️ **Le téléchargement prend 10-20 minutes** selon ta connexion.
> 💾 **Espace disque nécessaire** : ~15 GB

In [ ]:
from ultralytics import YOLO
import os

# Créer le dossier de travail
WORK_DIR = "/content/retail-heros-training"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print(f"📁 Dossier de travail: {WORK_DIR}")
print("🔄 Préparation du dataset SKU110K...")
print("   (Le téléchargement commencera automatiquement à l'entraînement)")

## 🏋️ Étape 3 : Entraîner le modèle

Configuration optimisée pour Retail-Héros.

| Paramètre | Valeur | Description |
|-----------|--------|-------------|
| `epochs` | 100 | Nombre d'itérations |
| `imgsz` | 640 | Taille des images |
| `batch` | 16 | Images par batch |
| `model` | yolov8n | Variante (n/s/m/l) |

> 🎛️ **Choix du modèle** :
> - `yolov8n` (nano) : Rapide, léger — pour mobile/edge
> - `yolov8s` (small) : Bon compromis — **recommandé**
> - `yolov8m` (medium) : Précision + — pour serveur
> - `yolov8l` (large) : Maximum précision — lent

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — Modifie ces paramètres selon tes besoins
# ═══════════════════════════════════════════════════════════════════════════════

MODEL_VARIANT = "yolov8s"      # yolov8n, yolov8s, yolov8m, yolov8l
EPOCHS = 100                    # Nombre d'epochs (50-300)
IMG_SIZE = 640                  # Taille des images (320, 416, 640, 1280)
BATCH_SIZE = 16                 # Dépend de ta mémoire GPU (8, 16, 32)

print("⚙️  Configuration:")
print(f"   Modèle: {MODEL_VARIANT}")
print(f"   Epochs: {EPOCHS}")
print(f"   Image size: {IMG_SIZE}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Dataset: SKU110K (11,743 images)")
print("\n🚀 Lancement de l'entraînement...")
print("   ⏱️  Durée estimée: 1-2 heures\n")

# Charger le modèle pré-entraîné
model = YOLO(f"{MODEL_VARIANT}.pt")

# Lancer l'entraînement
results = model.train(
    data="SKU110K.yaml",
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    name="retail-heros",
    project=WORK_DIR,
    
    # Early stopping
    patience=20,
    
    # Sauvegarde
    save=True,
    save_period=10,
    
    # Optimiseur
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    
    # Augmentation de données (optimisé pour rayons)
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    
    # Logs
    verbose=True
)

## 📊 Étape 4 : Visualiser les résultats

Affichage des métriques d'entraînement et des exemples de détection.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import glob

RESULTS_DIR = f"{WORK_DIR}/retail-heros"

# Afficher les courbes d'entraînement
print("📈 Courbes d'entraînement:")
for img_path in glob.glob(f"{RESULTS_DIR}/*.png"):
    display(Image.open(img_path))

# Métriques finales
print("\n📊 Résultats finaux:")
print(f"   mAP@0.5:    {results.results_dict.get('metrics/mAP50(B)', 'N/A'):.4f}")
print(f"   mAP@0.5:0.95: {results.results_dict.get('metrics/mAP50-95(B)', 'N/A'):.4f}")
print(f"   Precision:  {results.results_dict.get('metrics/precision(B)', 'N/A'):.4f}")
print(f"   Recall:     {results.results_dict.get('metrics/recall(B)', 'N/A'):.4f}")

## 🧪 Étape 5 : Tester le modèle sur une image

Inférence sur une image de test pour vérifier la qualité.

In [ ]:
# Charger le meilleur modèle
best_model = YOLO(f"{RESULTS_DIR}/weights/best.pt")

# Télécharger une image de test (rayon de supermarché)
!wget -q "https://images.unsplash.com/photo-1604719312566-8912e9227c6a?w=640" -O test_shelf.jpg

# Inférence
results_test = best_model("test_shelf.jpg", conf=0.3)

# Afficher le résultat
for r in results_test:
    im_array = r.plot()
    im = Image.fromarray(im_array[..., ::-1])
    display(im)
    print(f"\n🔍 {len(r.boxes)} produits détectés")
    print(f"   Confiance moyenne: {r.boxes.conf.mean().item():.3f}")

## 💾 Étape 6 : Télécharger le modèle pour Retail-Héros

Télécharge le modèle entraîné et utilise-le dans ton application Retail-Héros.

In [ ]:
from google.colab import files
import shutil

# Renommer le modèle au format Retail-Héros
model_source = f"{RESULTS_DIR}/weights/best.pt"
model_dest = "/content/retail-heros-yolo.pt"

shutil.copy2(model_source, model_dest)

print("📦 Modèle prêt pour Retail-Héros !")
print(f"   Source: {model_source}")
print(f"   Taille: {os.path.getsize(model_dest) / 1024**2:.1f} MB\n")

# Télécharger
print("⬇️  Téléchargement en cours...")
files.download(model_dest)

print("\n✅ Instructions d'utilisation:")
print("   1. Place le fichier dans retail-heros/models/")
print("   2. Renomme-le en 'retail-heros-yolo.pt'")
print("   3. Relance Retail-Héros — il utilisera automatiquement ton modèle !")

## 🎯 Étape 7 (Optionnel) : Fine-tune sur tes propres images

Si tu as tes propres photos de rayons, tu peux fine-tuner le modèle pour + de précision.

### 7.1 Upload tes images

In [ ]:
# Upload tes images (optionnel)
# from google.colab import files
# uploaded = files.upload()
# for filename in uploaded.keys():
#     print(f'Uploadé: {filename}')

print("💡 Pour fine-tuner sur tes images:")
print("   1. Annoter tes images avec Roboflow (roboflow.com)")
print("   2. Exporter en format YOLOv8")
print("   3. Upload le dataset ici")
print("   4. Relancer l'entraînement avec data='ton_dataset.yaml'")

## 📚 Récapitulatif

| Étape | Action | Résultat |
|-------|--------|----------|
| 0 | Vérifier GPU | GPU Tesla T4 activé |
| 1 | Installer | Ultralytics YOLOv8 |
| 2 | Télécharger | Dataset SKU110K (13.6 GB) |
| 3 | Entraîner | Modèle YOLO personnalisé |
| 4 | Visualiser | Courbes et métriques |
| 5 | Tester | Inférence sur image |
| 6 | Télécharger | `retail-heros-yolo.pt` |

---

## 🔗 Liens utiles

- **Retail-Héros GitHub** : https://github.com/TON_USER/retail-heros
- **SKU110K Dataset** : https://github.com/eg4000/SKU110K_CVPR19
- **Ultralytics Docs** : https://docs.ultralytics.com
- **Roboflow** : https://universe.roboflow.com (pour annoter tes images)

---

**Retail-Héros** — *Entraîné sur GPU, déployé partout.* 🚀